In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
#
import pandas as pd
import numpy as np
from pathlib import Path
from utils.benchmarks import BENCHMARKS_TO_PATHS


In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_0.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 0 ###

benchmark_name = "imdb"
filename = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "movie_metadata.csv"

m = pd.read_csv(filename)
factor = 10
m = pd.concat([m] * factor)

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_1.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 1 ###

m.info()

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 2 ###

m.movie_title = m.movie_title.str.strip()
m.duration = pd.to_numeric(m.duration)
m.budget = pd.to_numeric(m.budget)
m.gross = pd.to_numeric(m.gross)
m.imdb_score = pd.to_numeric(m.imdb_score)
m.set_index("movie_title", inplace=True)
m.drop(
    [
        "color",
        "director_facebook_likes",
        "actor_3_facebook_likes",
        "actor_2_name",
        "actor_1_facebook_likes",
        "actor_1_name",
    ],
    axis=1,
    inplace=True,
)
m.drop(
    [
        "cast_total_facebook_likes",
        "movie_imdb_link",
        "language",
        "actor_2_facebook_likes",
        "aspect_ratio",
    ],
    axis=1,
    inplace=True,
)
m.drop(
    ["actor_3_name", "facenumber_in_poster", "plot_keywords", "country"],
    axis=1,
    inplace=True,
)

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_3.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 3 ###

m.columns

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_4.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 4 ###

ninety_min_num_movies = len(m.duration[m.duration > 90.0])
total_num_movies = len(m.index[m.duration != np.nan])
ninety_min_num_movies / total_num_movies

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_5.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 5 ###

two_hour_movies = len(m.duration[m.duration > 120.0])
two_hour_movies / total_num_movies

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_6.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 6 ###

num_movies_directed = len(m.director_name[m.director_name != np.nan])
spiel = len(m.director_name[m.director_name == "Steven Spielberg"])
spiel / num_movies_directed

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_7.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 7 ###

e_movies = m[m.director_name == "Clint Eastwood"]
e_gross_under_budget = len(
    e_movies[
        (e_movies.gross != np.nan)
        & (e_movies.budget != np.nan)
        & (e_movies.gross < e_movies.budget)
    ]
)
e_gross_under_budget / len(e_movies.index)

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_8.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 8 ###

movies_with_budget_and_gross = m[(~pd.isnull(m.gross)) & (~pd.isnull(m.budget))]
gross_over_budget = m[
    (m.gross > m.budget) & (~pd.isnull(m.gross)) & (~pd.isnull(m.budget))
]

len(gross_over_budget) / len(movies_with_budget_and_gross)

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_9.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 9 ###

average_gross = m.gross.mean()
movie_grossed_over_average = len(
    m.index[(~pd.isnull(m.gross)) & (m.gross > average_gross)]
)
total_movie_with_gross = len(m.index[~pd.isnull(m.gross)])
movie_grossed_over_average / total_movie_with_gross

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 10 ###

movies_with_scores = m[
    (~pd.isnull(m.imdb_score) & (~pd.isnull(m.gross)) & (~pd.isnull(m.budget)))
][["imdb_score", "gross", "budget"]]
positive_scores = movies_with_scores[movies_with_scores.imdb_score > 6]
false_positives = positive_scores[positive_scores.gross < positive_scores.budget]
len(false_positives) / len(positive_scores)

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_11.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 11 ###

negative_scores = movies_with_scores[movies_with_scores.imdb_score <= 6]
false_negatives = negative_scores[negative_scores.gross > negative_scores.budget]

len(false_negatives) / len(negative_scores)

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_12.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 12 ###

scores = m.imdb_score

# plt.figure(figsize=(10, 3))
# plt.hist(scores, bins=np.arange(1, 11))
# plt.title("Distribution of Ratings")
# plt.xlabel("IMDB Rating")
# plt.ylabel("# of Movies")
# plt.show()

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_13.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 13 ###

scores.describe()

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/pre_cell_14.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 14 ###

mean = scores.mean()
median = scores.quantile(0.5)
std = scores.std()
mean, median, std